In [2]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import sys
import os
from pathlib import Path

REPO_ROOT = Path('/Users/dhillo/Garage/codeComplete/full measure/warfarin/')

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))



In [3]:
from src.bandits.epsilon_greedy_bandit import *
from src.bandits.linUCB import *
from src.bandits.thompson_sampling import *

In [5]:
# Load the dataset
df = pd.read_csv('output/warfarin_one_hot_encoded_full_power.csv')
# Get the number of arms from the number of unique labels
n_arms = df['Therapeutic Dose of Warfarin'].nunique()
df.shape, n_arms


((5528, 168), 3)

In [9]:
for c in df.columns:
    print(c)

Height (cm)
Weight (kg)
Therapeutic Dose of Warfarin
Acetaminophen or Paracetamol (Tylenol)_0.0
Acetaminophen or Paracetamol (Tylenol)_1.0
Age_1
Age_2
Age_3
Age_4
Age_5
Age_6
Age_7
Age_8
Age_9
Amiodarone (Cordarone)_0.0
Amiodarone (Cordarone)_1.0
Anti-fungal Azoles_0.0
Anti-fungal Azoles_1.0
Aspirin_0.0
Aspirin_1.0
Atorvastatin (Lipitor)_0.0
Atorvastatin (Lipitor)_1.0
CYP2C9 consensus_*1/*1
CYP2C9 consensus_*1/*11
CYP2C9 consensus_*1/*13
CYP2C9 consensus_*1/*14
CYP2C9 consensus_*1/*2
CYP2C9 consensus_*1/*3
CYP2C9 consensus_*1/*5
CYP2C9 consensus_*1/*6
CYP2C9 consensus_*2/*2
CYP2C9 consensus_*2/*3
CYP2C9 consensus_*3/*3
CYP2C9 consensus_Unknown
Carbamazepine (Tegretol)_False
Carbamazepine (Tegretol)_True
Cerivastatin (Baycol)_0.0
Combined QC CYP2C9_*1/*1
Combined QC CYP2C9_*1/*2
Combined QC CYP2C9_*1/*3
Combined QC CYP2C9_*2/*2
Combined QC CYP2C9_*2/*3
Combined QC CYP2C9_*3/*3
Congestive Heart Failure and/or Cardiomyopathy_0.0
Congestive Heart Failure and/or Cardiomyopathy_1.0
Current S

In [7]:
df.head

<bound method NDFrame.head of       Height (cm)  Weight (kg)  Therapeutic Dose of Warfarin  \
0          193.04        115.7                             1   
1          176.53        144.2                             1   
2          162.56         77.1                             2   
3          182.24         90.7                             1   
4          167.64         72.6                             1   
...           ...          ...                           ...   
5523       185.42        113.6                             2   
5524       160.02         55.9                             1   
5525       187.96         97.7                             2   
5526       177.80         87.3                             2   
5527       190.50         79.6                             1   

      Acetaminophen or Paracetamol (Tylenol)_0.0  \
0                                          False   
1                                          False   
2                                          Fa

# epsilon greedy bandit

In [5]:
# Set up the simulation
n_iterations = 20
cumulative_regrets = []
average_accuracies = []


for i in range(n_iterations):
    # Shuffle the dataset
    df_shuffled = shuffle(df, random_state=i)
    
    # Split the data into features and labels
    X = df_shuffled.drop('Therapeutic Dose of Warfarin', axis=1).values
    y = df_shuffled['Therapeutic Dose of Warfarin'].values
    
    # Initialize the bandit
    bandit = EpsilonGreedyBandit(n_arms)
    
    # Track performance
    correct_predictions = 0
    cumulative_regret = 0
    
    # Simulate the bandit selecting arms and receiving reward
    for j in range(len(df_shuffled)):
        # The bandit makes a prediction (selects an arm)
        chosen_arm = bandit.select_arm()
        
        # Get the actual label
        actual_label = y[j]
        
        # Check if the bandit's prediction was correct
        if chosen_arm == actual_label:
            reward = 1
            correct_predictions += 1
        else:
            reward = 0
            
        # Update the bandit with the reward for the chosen arm
        bandit.update(chosen_arm, reward)
        
        # Calculate regret (difference between the optimal and chosen action)
        optimal_reward = 1  # assuming the optimal action would always be correct
        regret = optimal_reward - reward
        cumulative_regret += regret
    
    # Store the cumulative regret for this iteration
    cumulative_regrets.append(cumulative_regret)
    
    # Calculate accuracy for this iteration
    accuracy = correct_predictions / len(df_shuffled)
    average_accuracies.append(accuracy)
    
    print(f"Iteration {i+1}: Accuracy = {accuracy}, Cumulative Regret = {cumulative_regret}")

# Calculate the average accuracy over all iterations
final_average_accuracy = np.mean(average_accuracies)
print(f"Average Accuracy over {n_iterations} iterations: {final_average_accuracy}")



Iteration 1: Accuracy = 0.5806801736613604, Cumulative Regret = 2318
Iteration 2: Accuracy = 0.5774240231548481, Cumulative Regret = 2336
Iteration 3: Accuracy = 0.5839363241678727, Cumulative Regret = 2300
Iteration 4: Accuracy = 0.5877351664254703, Cumulative Regret = 2279
Iteration 5: Accuracy = 0.5734442836468886, Cumulative Regret = 2358
Iteration 6: Accuracy = 0.5830318379160637, Cumulative Regret = 2305
Iteration 7: Accuracy = 0.5779667149059334, Cumulative Regret = 2333
Iteration 8: Accuracy = 0.5794138929088278, Cumulative Regret = 2325
Iteration 9: Accuracy = 0.5786903039073806, Cumulative Regret = 2329
Iteration 10: Accuracy = 0.5743487698986975, Cumulative Regret = 2353
Iteration 11: Accuracy = 0.5738060781476122, Cumulative Regret = 2356
Iteration 12: Accuracy = 0.5833936324167872, Cumulative Regret = 2303
Iteration 13: Accuracy = 0.5828509406657019, Cumulative Regret = 2306
Iteration 14: Accuracy = 0.578328509406657, Cumulative Regret = 2331
Iteration 15: Accuracy = 0.583

# lin UCB

In [3]:

# Define parameters
alpha = 1.0  # Exploration parameter
n_iterations = 20
cumulative_regrets = []
average_accuracies = []

# Get the number of arms and features
n_arms = df['Therapeutic Dose of Warfarin'].nunique()
n_features = df.shape[1] - 1  # Number of features is total columns minus the label column

for i in range(n_iterations):
    df_shuffled = shuffle(df, random_state=i)
    X = df_shuffled.drop('Therapeutic Dose of Warfarin', axis=1).values
    y = df_shuffled['Therapeutic Dose of Warfarin'].values
    
    # Initialize the LinUCB model
    linucb = LinUCB(alpha, n_arms, n_features)
    
    correct_predictions = 0
    cumulative_regret = 0

    for j in tqdm(range(len(df_shuffled)), desc=f'Iteration {i+1}'):
        x = X[j]
        chosen_arm = linucb.select_arm(x)
        actual_label = y[j]
        
        reward = 1 if chosen_arm == actual_label else 0
        correct_predictions += reward
        
        linucb.update(chosen_arm, x, reward)
        
        optimal_reward = 1  # assuming the optimal action would always be correct
        regret = optimal_reward - reward
        cumulative_regret += regret
    
    cumulative_regrets.append(cumulative_regret)
    
    accuracy = correct_predictions / len(df_shuffled)
    average_accuracies.append(accuracy)
    
    print(f"Iteration {i+1}: Accuracy = {accuracy}, Cumulative Regret = {cumulative_regret}")

final_average_accuracy = np.mean(average_accuracies)
print(f"Average Accuracy over {n_iterations} iterations: {final_average_accuracy}")


Iteration 1: 100%|██████████| 5528/5528 [00:52<00:00, 105.81it/s]


Iteration 1: Accuracy = 0.6338639652677279, Cumulative Regret = 2024


Iteration 2: 100%|██████████| 5528/5528 [00:51<00:00, 107.52it/s]


Iteration 2: Accuracy = 0.6293415340086831, Cumulative Regret = 2049


Iteration 3: 100%|██████████| 5528/5528 [00:53<00:00, 103.06it/s]


...

Iteration 18: Accuracy = 0.6414616497829233, Cumulative Regret = 1982


Iteration 19: 100%|██████████| 5528/5528 [00:54<00:00, 101.54it/s]


Iteration 19: Accuracy = 0.6333212735166426, Cumulative Regret = 2027


Iteration 20: 100%|██████████| 5528/5528 [01:01<00:00, 90.53it/s] 

Iteration 20: Accuracy = 0.6400144717800289, Cumulative Regret = 1990
Average Accuracy over 20 iterations: 0.6353020984081043


# Thompson sampling

In [3]:
# Define parameters
n_iterations = 20
cumulative_regrets = []
average_accuracies = []

# Get the number of arms from the number of unique labels
n_arms = df['Therapeutic Dose of Warfarin'].nunique()

for i in range(n_iterations):
    df_shuffled = shuffle(df, random_state=i)
    X = df_shuffled.drop('Therapeutic Dose of Warfarin', axis=1).values
    y = df_shuffled['Therapeutic Dose of Warfarin'].values
    
    # Initialize the Thompson Sampling bandit
    ts_bandit = ThompsonSamplingBandit(n_arms)
    
    correct_predictions = 0
    cumulative_regret = 0
    
    # Simulate the decision process
    for j in range(len(df_shuffled)):
        chosen_arm = ts_bandit.select_arm()
        actual_label = y[j]
        
        reward = 1 if chosen_arm == actual_label else 0
        correct_predictions += reward
        
        ts_bandit.update(chosen_arm, reward)
        
        optimal_reward = 1  # assuming the optimal action would always be correct
        regret = optimal_reward - reward
        cumulative_regret += regret
    
    cumulative_regrets.append(cumulative_regret)
    
    accuracy = correct_predictions / len(df_shuffled)
    average_accuracies.append(accuracy)
    
    print(f"Iteration {i+1}: Accuracy = {accuracy}, Cumulative Regret = {cumulative_regret}")

final_average_accuracy = np.mean(average_accuracies)
print(f"Average Accuracy over {n_iterations} iterations: {final_average_accuracy}")

Iteration 1: Accuracy = 0.6094428364688856, Cumulative Regret = 2159
Iteration 2: Accuracy = 0.6085383502170767, Cumulative Regret = 2164
Iteration 3: Accuracy = 0.6094428364688856, Cumulative Regret = 2159
Iteration 4: Accuracy = 0.6092619392185239, Cumulative Regret = 2160
Iteration 5: Accuracy = 0.6092619392185239, Cumulative Regret = 2160
Iteration 6: Accuracy = 0.6098046309696092, Cumulative Regret = 2157
Iteration 7: Accuracy = 0.6094428364688856, Cumulative Regret = 2159
Iteration 8: Accuracy = 0.6096237337192475, Cumulative Regret = 2158
Iteration 9: Accuracy = 0.608357452966715, Cumulative Regret = 2165
Iteration 10: Accuracy = 0.6096237337192475, Cumulative Regret = 2158
Iteration 11: Accuracy = 0.6092619392185239, Cumulative Regret = 2160
Iteration 12: Accuracy = 0.6094428364688856, Cumulative Regret = 2159
Iteration 13: Accuracy = 0.6096237337192475, Cumulative Regret = 2158
Iteration 14: Accuracy = 0.6096237337192475, Cumulative Regret = 2158
Iteration 15: Accuracy = 0.609

# finding accuracies for the clinical algorithm and the pharmacogenetic algorithm

In [4]:
%pwd

'/Users/dhillo/Garage/codeComplete/full measure/warfarin/src/bandits'

In [5]:
df_with_baselines = pd.read_csv('./../../data/warfarin_with_baselines.csv')

In [6]:
df_with_baselines.shape

(5528, 74)

In [7]:
df_with_baselines.head

<bound method NDFrame.head of      PharmGKB Subject ID  Gender   Race               Ethnicity  Age  \
0            PA135312261    male  White  not Hispanic or Latino    6   
1            PA135312262  female  White  not Hispanic or Latino    5   
2            PA135312263  female  White  not Hispanic or Latino    4   
3            PA135312264    male  White  not Hispanic or Latino    6   
4            PA135312265    male  White  not Hispanic or Latino    5   
...                  ...     ...    ...                     ...  ...   
5523         PA152407681    male  White  not Hispanic or Latino    2   
5524         PA152407682  female  White  not Hispanic or Latino    7   
5525         PA152407683    male  White  not Hispanic or Latino    6   
5526         PA152407684    male  White  not Hispanic or Latino    6   
5527         PA152407685    male  White  not Hispanic or Latino    7   

      Height (cm)  Weight (kg) Indication for Warfarin Treatment  \
0          193.04        115.7       

In [8]:
df = df_with_baselines.copy()

In [11]:
df['Clinical Correct'] = df['Clinical Dose'] == df["Therapeutic Dose of Warfarin"]
accuracy_clinical = df['Clinical Correct'].sum() / len(df['Clinical Correct'])
print("df['Clinical Correct'].value_counts()")
print(df['Clinical Correct'].value_counts())
print("Clinical Accuracy:",accuracy_clinical)

df['Clinical Correct'].value_counts()
Clinical Correct
True     3542
False    1986
Name: count, dtype: int64
Clinical Accuracy: 0.6407380607814761


In [12]:
df['Pharmacogenetic Correct'] = df['Pharmacogenetic Dose'] == df["Therapeutic Dose of Warfarin"]
accuracy_pharmacogenetic = df['Pharmacogenetic Correct'].sum() / len(df['Pharmacogenetic Correct'])
print("df['Pharmacogenetic Correct'].value_counts()")
print(df['Pharmacogenetic Correct'].value_counts())
print("Pharmacogenetic Accuracy:",accuracy_pharmacogenetic)


df['Pharmacogenetic Correct'].value_counts()
Pharmacogenetic Correct
True     3814
False    1714
Name: count, dtype: int64
Pharmacogenetic Accuracy: 0.6899421128798843


In [10]:
df['Fixed Correct'] = df['Fixed Dose'] == df["Therapeutic Dose of Warfarin"]
accuracy_fixed = df['Fixed Correct'].sum() / len(df['Fixed Correct'])
print("df['Fixed Correct'].value_counts()")
print(df['Fixed Correct'].value_counts())
print("Fixed Accuracy:",accuracy_fixed)

df['Fixed Correct'].value_counts()
Fixed Correct
True     3382
False    2146
Name: count, dtype: int64
Fixed Accuracy: 0.611794500723589
